# 12.6 怎麼組合頻率區段？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：從聽到的起伏，找到每一小段的頻率**

frame→window→STFT→mel→log，每次只加一個步驟。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/audio.svg")))

**先想一想：**bands從16變32，時間框數會加倍嗎？

mel軸在低頻較密、高頻較疏；一組重疊三角濾波器把細頻率格加總成較少bands。它是前處理座標選擇，不是學來的編碼器。

**把直覺寫成數學：**mel(f)=2595·log10(1+f/700)；過濾矩陣[M,F]乘power[F,T]。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.multimodal import mel_filter_bank

bank = mel_filter_bank(bands=16)
print(bank.shape, bank.min().item(), bank.max().item())
assert bank.shape == (16, 201) and (bank >= 0).all()

**如何讀結果：**每個band都有自己的頻率權重；高低頻並非簡單等寬分組。

**只改一件事：**只把bands改爲32。
